# AVG Distance of nieghbors

In [1]:
import os
import sys
import shutil
import math
import warnings
import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import *
from enum import Enum
from scipy.spatial import Voronoi
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wasserstein_distance
sys.path.append("/home/esraan/CellDeathSpreading/")
from src.utils import get_experiment_cell_death_times_by_specific_siliding_window,read_experiment_cell_xy_and_death_times
from src.NucleationAndPropagationMeasurements import replace_ugly_long_name
from src.quanta_utils import get_neighbors
from src.DeathQuanta import DeathQuanta
from src.NucleationAndPropagationMeasurements import *
import seaborn as sns
from src.uSpiCalc import uSpiCalc
from src.mSpiCalc import mSpiCalc
from src.mixSpiCalc import mixSpiCalc

In [2]:

def simple_treatment(name):
    # if "field" in name.lower():
    #     return "FB"
    if "nec" in name.lower():
        return "NecrosisInColonies"
    elif "apop" in name.lower():
        return "ApoptosisInColonies"
    else:
        if "_A" in name:
            return "MixedSubApop"
        elif "_N" in name:
            return "MixedSubNec"
        return "MixedColony"

In [3]:
exps_dir_name = "/sise/assafzar-group/assafzar/Esraa/SalJyoPaperFiles/mixed_pure_death_minutes_microns"
meta_data_file_full_path= "/sise/assafzar-group/assafzar/Esraa/SalJyoPaperFiles/all_csv_files_minutes_microns_with_sytox.csv"# "/sise/assafzar-group/assafzar/Esraa/Others/UpdatedMetaData.csv"
meta_data_extract_exp_names= pd.read_csv(meta_data_file_full_path)
exp_names = meta_data_extract_exp_names.iloc[:,1]
all_previous_experiments_spi_and_ni_regeneration = calc_all_experiments_SPI_and_NI_for_landscape(list(exp_names),exps_dir_path=exps_dir_name,
                                                                                                 meta_data_full_file_path=meta_data_file_full_path,
                                                                                                 dist_threshold=100,
                                                                                                 filter_neighbors_by_distance = 1,
                                                                                                 filter_neighbors_by_level = 1,)
reformatting_all_previos_experiments_spi_and_ni_regeneration = {"Experiment_name":[],
                                                                "SPI":[],
                                                                "NI":[],
                                                                "Treatment":[],
                                                                "Cell Line + Treatment":[],
                                                                "Cell Line":[],
                                                                "Origin":[],
                                                                "Mode":[],
                                                                "Density":[],
                                                                "pvalue":[],
                                                                "MeanDistance": []}
for key,value in all_previous_experiments_spi_and_ni_regeneration.items():
    if value is None or "--19" in key or "mixed" in simple_treatment(key):
        continue
    origin = meta_data_extract_exp_names[meta_data_extract_exp_names["File Name"] == key]["Origin"].values[0]
    mode = meta_data_extract_exp_names[meta_data_extract_exp_names["File Name"] == key]["Mode"].values[0]
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Origin"].append(origin)
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Mode"].append(mode)
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Experiment_name"].append(key)
    exp_cell_line= meta_data_extract_exp_names[meta_data_extract_exp_names["File Name"]== key]["Cell Line"].values[0]
    reformatting_all_previos_experiments_spi_and_ni_regeneration["SPI"].append(value[0])
    reformatting_all_previos_experiments_spi_and_ni_regeneration["NI"].append(value[1])
    reformatting_all_previos_experiments_spi_and_ni_regeneration["pvalue"].append(value[2])
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Cell Line"].append(exp_cell_line)
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Cell Line + Treatment"].append(replace_ugly_long_name(key,exp_cell_line))
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Treatment"].append(simple_treatment(key))
    density = meta_data_extract_exp_names[meta_data_extract_exp_names["File Name"] == key]["Density(#Cells)"].values[0]
    reformatting_all_previos_experiments_spi_and_ni_regeneration['MeanDistance'].append(value[3])
    reformatting_all_previos_experiments_spi_and_ni_regeneration["Density"].append(density)

In [9]:
filtered_df = pd.DataFrame(reformatting_all_previos_experiments_spi_and_ni_regeneration)
filtered_df = filtered_df[~filtered_df['Experiment_name'].str.contains('pure|colony|SKT|outl|ROI#2|mixed', case=False, na=False)]

In [12]:
filtered_df.iloc[:, [0, -1]].to_csv('output.csv', index=False)